In [ ]:
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate

load_dotenv()  # 加载.env文件里的变量

llm = ChatOpenAI(
    model="deepseek-chat",  # 使用的模型名称，目前官方推荐用 'deepseek-chat'
    api_key=os.getenv("DEEPSEEK_API_KEY"),  # 你的 DeepSeek API Key
    base_url="https://api.deepseek.com/v1",  # DeepSeek API 地址
    temperature=0,
)

In [ ]:
from langgraph.graph import StateGraph, START, END, MessagesState
from langchain_core.messages import HumanMessage, SystemMessage

# MessageGraph 已在 langgraph 1.0 弃用，改用 StateGraph + MessagesState
# MessagesState 内置了 messages 键，并使用 add_messages reducer 自动累加消息
builder = StateGraph(MessagesState)

def chatbot(state):
    print(state)
    return {'messages': [llm.invoke(state['messages'])]}


builder.add_node('chatbot', chatbot)

builder.add_edge(START, 'chatbot')
builder.add_edge('chatbot', END)

graph = builder.compile()

In [ ]:
from IPython.display import display,Image

display(Image(graph.get_graph(xray=True).draw_mermaid_png()))

In [ ]:
initial_state={'messages':['请详细的介绍一下你自己']}
result=graph.invoke(initial_state)
print(result['messages'][-1].content)

In [ ]:
print(result)

In [ ]:
def stream_graph_updates(user_input:str):
    for event in graph.stream({'messages':[('user',user_input)]}):
        for value in event.values():
            print("模型回复:",value['messages'][-1].content)

while True:
    try:
        user_input=input("用户提问：")
        if user_input.lower() in ['退出']:
            print("下次再见")
            break
        
        stream_graph_updates(user_input)
    except:
        break